# A Minimal Tritium Fuel Cycle

The simplest closed tritium fuel cycle has four components: a storage that holds the fuel, the plasma that burns a small part of it, a breeding blanket that breeds new tritium, and an extraction system that recovers it. This example connects one block for each of them into a loop and follows the tritium inventories from startup.

Each component is a residence time model. It holds one inventory $I$ that is filled by its inflows and leaves towards the next component after the residence time $\tau$, while it decays with the constant $\lambda$:

$$\frac{dI}{dt} = \sum_i u_i - \frac{I}{\tau} - \lambda I$$

The loop is:

- the fuelling takes tritium from the **storage** and injects it into the **plasma**
- the fraction $f_b$ of what leaves the plasma was burnt, the rest returns to the storage
- every burnt tritium atom breeds $TBR$ new ones in the **breeding blanket**
- the **tritium extraction system** recovers the fraction $\eta$ of what it receives and sends it to the storage, the rest goes back to the blanket

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("../pathsim_docs.mplstyle")

from pathsim import Connection, Simulation
from pathsim.blocks import Amplifier, Constant, Scope, Splitter
from pathsim.solvers import RKCK54

from pathsim_fusion import (
    BreedingBlanket,
    Plasma,
    StorageAndManagement,
    TritiumExtractionSystem,
)

## Parameters

The values are illustrative. Inventories are in kg and times in seconds.

In [ ]:
HOUR = 3600.0
DAY = 24 * HOUR

fuelling_rate = 4.5e-5  # tritium injected into the plasma [kg/s]
burn_fraction = 0.02  # fraction of the injected tritium that is burnt [-]
tbr = 1.07  # tritium breeding ratio [-]
efficiency = 0.95  # extraction efficiency [-]

tau_plasma = 0.5 * HOUR
tau_blanket = 1.25 * HOUR
tau_extraction = 24 * HOUR

decay_constant = 1.78e-9  # tritium decay constant [1/s]
startup_inventory = 1.0  # tritium in the storage at the start [kg]

## Model

The four components are blocks of `pathsim_fusion`. The storage has no residence time: it accumulates what flows in, and the fuelling is connected to it as a negative inflow. Three core blocks complete the loop: a `Constant` for the fuelling rate, a `Splitter` that divides the flow leaving the plasma into the burnt and the unburnt part, and an `Amplifier` that turns the burn rate into the breeding rate.

In [ ]:
# components
storage = StorageAndManagement(
    decay_constant=decay_constant, initial_value=startup_inventory
)
plasma = Plasma(tau=tau_plasma, decay_constant=decay_constant)
blanket = BreedingBlanket(tau=tau_blanket, decay_constant=decay_constant)
extraction = TritiumExtractionSystem(
    tau=tau_extraction, efficiency=efficiency, decay_constant=decay_constant
)

# fuelling: taken from the storage, injected into the plasma
fuelling = Constant(fuelling_rate)
withdrawal = Amplifier(-1.0)

# what leaves the plasma is either burnt or returns to the storage
burn = Splitter([burn_fraction, 1 - burn_fraction])
breeding = Amplifier(tbr)

sco = Scope(labels=["storage", "plasma", "blanket", "extraction"])

blocks = [
    storage,
    plasma,
    blanket,
    extraction,
    fuelling,
    withdrawal,
    burn,
    breeding,
    sco,
]

connections = [
    # storage -> plasma
    Connection(fuelling, plasma[0], withdrawal),
    Connection(withdrawal, storage[0]),
    # plasma -> blanket (bred) and storage (unburnt)
    Connection(plasma["outflow"], burn),
    Connection(burn[0], breeding),
    Connection(breeding, blanket[0]),
    Connection(burn[1], storage[1]),
    # blanket -> extraction -> storage
    Connection(blanket["outflow"], extraction[0]),
    Connection(extraction["extracted"], storage[2]),
    Connection(extraction["residual"], blanket[1]),
    # recorded inventories
    Connection(storage["inventory"], sco[0]),
    Connection(plasma["inventory"], sco[1]),
    Connection(blanket["inventory"], sco[2]),
    Connection(extraction["inventory"], sco[3]),
]

sim = Simulation(
    blocks,
    connections,
    Solver=RKCK54,
    dt=1.0,
    tolerance_lte_rel=1e-6,
    tolerance_lte_abs=1e-10,
    log=True,
)

sim.run(30 * DAY)

## Inventories

The plasma, the blanket and the extraction system fill up from empty and settle at a steady inventory, the slower the longer their residence time.

In [ ]:
time, (i_storage, i_plasma, i_blanket, i_extraction) = sco.read()
days = time / DAY

fig, ax = plt.subplots()
ax.plot(days, i_plasma, label="plasma")
ax.plot(days, i_blanket, label="breeding blanket")
ax.plot(days, i_extraction, label="tritium extraction")
ax.set_yscale("log")
ax.set_ylim(1e-4, 1)
ax.set_xlabel("time [days]")
ax.set_ylabel("tritium inventory [kg]")
ax.legend()
plt.show()

The storage shows the behaviour that sizes a fuel cycle. At first it only supplies fuel, because the bred tritium is still on its way through the blanket and the extraction system. Once that tritium arrives, the inventory recovers and then grows, since more tritium is bred than burnt. The lowest point sets the tritium that has to be available at startup.

In [ ]:
fig, ax = plt.subplots()
ax.plot(days, i_storage)
ax.axhline(startup_inventory, color="grey", ls="--", lw=1)
ax.set_xlabel("time [days]")
ax.set_ylabel("storage inventory [kg]")
plt.show()

i_min = i_storage.min()
print(
    f"lowest storage inventory: {i_min:.3f} kg after {days[i_storage.argmin()]:.1f} days"
)
print(f"tritium tied up in the cycle: {startup_inventory - i_min:.3f} kg")

In steady operation the storage gains the net breeding, $(TBR - 1)$ times the burn rate, minus the decay of all inventories:

In [ ]:
burn_rate = burn_fraction * fuelling_rate
print(f"net breeding: {(tbr - 1) * burn_rate * DAY * 1e3:.2f} g/day")

last_day = days >= 29
growth = (i_storage[-1] - i_storage[last_day][0]) / (days[-1] - days[last_day][0])
print(f"growth of the storage inventory at the end: {growth * 1e3:.2f} g/day")

Each block also has a `loss` output with its decay and non-radioactive losses, which can be integrated to close the tritium balance. A fuel cycle with all components of a power plant is built the same way, see the ARC fuel cycle example.